# Building a Production-Grade Weather Tool for LangChain Agents

## Overview

In this comprehensive tutorial, you'll learn how to build a **production-grade weather tool** that can be used by LangChain agents. This goes beyond basic API integration to implement enterprise-ready patterns for reliability, security, and observability.

## Learning Objectives

By the end of this tutorial, you will master the six key requirements for production-grade tool development:

1. **Input Validation and Type Safety**: Use Pydantic models to validate and sanitize all inputs before processing
2. **Error Handling Strategy**: Implement comprehensive error classification and provide clear, actionable error messages
3. **Resilience and Retry Mechanisms**: Handle transient failures gracefully with exponential backoff and jitter
4. **Observability and Monitoring**: Add structured logging to track tool performance and debug issues
5. **Security Considerations**: Sanitize inputs, protect API keys, and prevent injection attacks
6. **Performance and Scalability**: Use connection pooling, timeouts, and efficient data structures

## What You'll Build

You'll create two interconnected weather tools:

- **Current Weather**: Get real-time weather for any location
- **Weather Forecast**: Get weather predictions for the next 2 days

These tools will work seamlessly with LangChain agents, allowing them to autonomously answer weather-related questions.

## Important Note About OpenWeatherMap API Tiers

This tutorial uses only **FREE tier** endpoints:
- ✅ **Current Weather API** - Included in free tier
- ✅ **5 Day / 3 Hour Forecast API** - Included in free tier
- ❌ **Historical Weather (Time Machine API)** - Requires paid subscription

We focus on the two free-tier endpoints to ensure the tutorial works with a free OpenWeatherMap account.

## Prerequisites

**API Keys Required:**
- **OpenAI API Key**: For the LangChain agent (https://platform.openai.com/api-keys)
- **OpenWeatherMap API Key**: For weather data (https://openweathermap.org/api) - Free tier available!

**Required Packages:**
```bash
pip install langchain langchain-openai langchain-core python-dotenv requests tenacity pydantic
```

**Setup Instructions:**

1. Create a `.env` file in your project directory
2. Add your API keys:
   ```
   OPENAI_API_KEY=your-openai-key-here
   OPENWEATHER_API_KEY=your-openweather-key-here
   ```
3. Sign up for a free OpenWeatherMap account at https://openweathermap.org/api
4. Generate an API key from your account dashboard

## Why Production-Grade Matters

Basic tool integration might work in development, but production environments demand:

- **Reliability**: Tools must handle network failures, rate limits, and API changes
- **Security**: Protect against malicious inputs and data leaks
- **Observability**: Track performance and debug issues quickly
- **User Experience**: Provide clear error messages instead of cryptic failures

This tutorial shows you how to implement all of these features systematically.

## Step 1: Setup and Imports

First, we'll import all necessary libraries and configure our environment. We'll use:

- **python-dotenv**: Securely load API keys from environment files
- **requests**: Make HTTP requests with session management
- **tenacity**: Implement retry logic with exponential backoff
- **pydantic**: Validate inputs with type safety
- **logging**: Add structured logging for observability
- **langchain**: Build the agent and tool decorators

In [16]:
import os
import logging
import time
from datetime import datetime, timedelta
from typing import Optional, Dict, Any
from collections import defaultdict

import requests
from dotenv import load_dotenv
from pydantic import BaseModel, Field, field_validator
from tenacity import (
    retry,
    stop_after_attempt,
    wait_exponential,
    retry_if_exception_type,
    before_sleep_log
)

from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

# Load environment variables from .env file
load_dotenv()

# Verify API keys are loaded
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found in environment variables")
if not os.getenv("OPENWEATHER_API_KEY"):
    raise ValueError("OPENWEATHER_API_KEY not found in environment variables")

print("All API keys loaded successfully!")
print("Required packages imported successfully!")

All API keys loaded successfully!
Required packages imported successfully!


## Step 2: Configure Structured Logging

**Why Logging Matters:**

In production environments, you need to:
- Track which tools are being called and with what parameters
- Measure execution times to identify performance bottlenecks
- Debug failures by understanding the sequence of events
- Monitor API usage to avoid rate limits

We'll configure Python's logging module with a structured format that includes timestamps, log levels, and contextual information.

In [17]:
# Configure logging with structured format
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

# Create a logger for our weather tools
logger = logging.getLogger('weather_tools')

# Test the logger
logger.info("Logging system initialized successfully")

print("\nLogging configured with INFO level")
print("You'll see structured logs for all tool operations below")

2025-11-12 16:41:41 - weather_tools - INFO - Logging system initialized successfully



Logging configured with INFO level
You'll see structured logs for all tool operations below


## Step 3: Build Geocoding Helper Function

**The Challenge:**

OpenWeatherMap's weather APIs require latitude and longitude coordinates, but users naturally want to query by city name ("London", "Tokyo", "New York"). We need a geocoding function to convert city names to coordinates.

**Production Considerations:**

- **Input Sanitization**: Clean and validate city names to prevent injection attacks
- **Error Handling**: Provide clear messages when cities aren't found
- **Logging**: Track geocoding requests for debugging
- **API Key Security**: Never log the API key itself

In [18]:
def geocode_city(city_name: str) -> tuple[float, float]:
    """
    Convert a city name to latitude and longitude coordinates using OpenWeatherMap Geocoding API.
    
    This function is essential for converting user-friendly city names into the coordinates
    required by the OpenWeatherMap weather APIs.
    
    Args:
        city_name (str): The name of the city to geocode (e.g., "London", "New York")
        
    Returns:
        tuple[float, float]: A tuple of (latitude, longitude)
        
    Raises:
        ValueError: If the city is not found or the API request fails
        
    Example:
        >>> lat, lon = geocode_city("London")
        >>> print(f"London is at {lat}, {lon}")
    """
    # Sanitize input: strip whitespace and validate
    city_name = city_name.strip()
    
    if not city_name:
        raise ValueError("City name cannot be empty")
    
    # Additional sanitization: remove potentially dangerous characters
    # Allow only letters, spaces, hyphens, and apostrophes (for cities like "O'Fallon")
    if not all(c.isalnum() or c in " -'," for c in city_name):
        raise ValueError(f"Invalid city name format: {city_name}")
    
    logger.info(f"Geocoding city: {city_name}")
    
    api_key = os.getenv("OPENWEATHER_API_KEY")
    url = "http://api.openweathermap.org/geo/1.0/direct"
    
    params = {
        "q": city_name,
        "limit": 1,  # Only get the top result
        "appid": api_key
    }
    
    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        
        data = response.json()
        
        if not data:
            logger.warning(f"City not found: {city_name}")
            raise ValueError(
                f"Could not find coordinates for '{city_name}'. "
                "Please check the spelling or try a different city name."
            )
        
        lat = data[0]["lat"]
        lon = data[0]["lon"]
        
        logger.info(f"Successfully geocoded {city_name} to ({lat}, {lon})")
        
        return lat, lon
        
    except requests.exceptions.Timeout:
        logger.error(f"Geocoding request timed out for city: {city_name}")
        raise ValueError("The geocoding service is taking too long to respond. Please try again.")
    
    except requests.exceptions.RequestException as e:
        logger.error(f"Geocoding request failed: {str(e)}")
        raise ValueError(f"Failed to geocode city: {str(e)}")

# Test the geocoding function
print("Testing geocoding function...")
try:
    lat, lon = geocode_city("London")
    print(f"\nSuccess! London coordinates: {lat}, {lon}")
except Exception as e:
    print(f"\nError: {e}")

2025-11-12 16:41:42 - weather_tools - INFO - Geocoding city: London
2025-11-12 16:41:42 - weather_tools - INFO - Successfully geocoded London to (51.5073219, -0.1276474)


Testing geocoding function...

Success! London coordinates: 51.5073219, -0.1276474


## Step 4: Define Pydantic Models for Input Validation

**Why Pydantic?**

Pydantic provides automatic validation and type checking for our inputs. This ensures:

- **Type Safety**: Inputs are the correct type before processing
- **Validation**: Custom validators catch invalid data early
- **Clear Error Messages**: Users get helpful feedback about what went wrong
- **Documentation**: Models serve as clear API contracts

**Our Validation Models:**

1. **WeatherLocation**: Validates city names for current weather
2. **ForecastWeatherInput**: Validates inputs for forecast queries (enforces 2 days)

Note how we enforce the "2 days" requirement at the validation layer, making it impossible to accidentally request the wrong time range.

In [19]:
class WeatherLocation(BaseModel):
    """
    Validates a location input for weather queries.
    
    Attributes:
        city (str): The city name to query (e.g., "London", "New York")
    """
    city: str = Field(
        ..., 
        min_length=1,
        max_length=100,
        description="The name of the city to get weather for"
    )
    
    @field_validator('city')
    @classmethod
    def validate_city(cls, v):
        """Sanitize and validate city name."""
        v = v.strip()
        
        if not v:
            raise ValueError("City name cannot be empty or just whitespace")
        
        # Allow only safe characters
        if not all(c.isalnum() or c in " -'," for c in v):
            raise ValueError(
                f"City name contains invalid characters. "
                f"Only letters, numbers, spaces, hyphens, and apostrophes are allowed."
            )
        
        return v


class ForecastWeatherInput(BaseModel):
    """
    Validates input for weather forecast queries.
    Enforces that we always query for exactly 2 days of forecast data.
    
    Attributes:
        city (str): The city name to query
        days (int): Number of forecast days (must be 2)
    """
    city: str = Field(
        ...,
        min_length=1,
        max_length=100,
        description="The name of the city"
    )
    days: int = Field(
        default=2,
        description="Number of forecast days to retrieve (must be 2)"
    )
    
    @field_validator('days')
    @classmethod
    def validate_days(cls, v):
        """Ensure we only query for 2 days of forecast data."""
        if v != 2:
            raise ValueError(
                f"Weather forecast is only available for 2 days. Got: {v} days"
            )
        return v
    
    @field_validator('city')
    @classmethod
    def validate_city(cls, v):
        """Sanitize and validate city name."""
        v = v.strip()
        if not v:
            raise ValueError("City name cannot be empty")
        if not all(c.isalnum() or c in " -'," for c in v):
            raise ValueError("City name contains invalid characters")
        return v

# Test the validation models
print("Testing Pydantic validation models...\n")

# Test valid input
try:
    location = WeatherLocation(city="London")
    print(f"Valid location: {location.city}")
except Exception as e:
    print(f"Validation error: {e}")

# Test invalid input (empty city)
try:
    location = WeatherLocation(city="   ")
    print(f"Valid location: {location.city}")
except Exception as e:
    print(f"\nCaught expected validation error for empty city: {e}")

# Test invalid days for forecast
try:
    forecast = ForecastWeatherInput(city="Paris", days=5)
    print(f"Valid forecast input: {forecast}")
except Exception as e:
    print(f"\nCaught expected validation error for wrong days: {e}")

print("\nValidation models working correctly!")

Testing Pydantic validation models...

Valid location: London

Caught expected validation error for empty city: 1 validation error for WeatherLocation
city
  Value error, City name cannot be empty or just whitespace [type=value_error, input_value='   ', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

Caught expected validation error for wrong days: 1 validation error for ForecastWeatherInput
days
  Value error, Weather forecast is only available for 2 days. Got: 5 days [type=value_error, input_value=5, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

Validation models working correctly!


## Step 5: Build the Weather API Client Class

**Why a Client Class?**

Instead of making raw API calls in each tool function, we create a centralized client class that handles:

1. **Connection Pooling**: Reuse HTTP connections for better performance
2. **Retry Logic**: Automatically retry failed requests with exponential backoff
3. **Timeout Handling**: Prevent hanging requests
4. **Structured Logging**: Track all API interactions
5. **Error Classification**: Distinguish between authentication, network, and API errors

**Key Features:**

- **Session Management**: Using `requests.Session()` for connection pooling
- **Tenacity Integration**: Automatic retries with exponential backoff and jitter
- **Execution Timing**: Track how long each API call takes
- **Security**: Never log API keys

In [20]:
class WeatherAPIClient:
    """
    Production-grade client for OpenWeatherMap API with built-in resilience,
    observability, and error handling.
    
    Features:
    - Connection pooling via requests.Session
    - Automatic retries with exponential backoff
    - Structured logging for observability
    - Comprehensive error handling and classification
    - Request timeout protection
    """
    
    def __init__(self, api_key: str, timeout: int = 10, max_retries: int = 3):
        """
        Initialize the Weather API client.
        
        Args:
            api_key (str): OpenWeatherMap API key
            timeout (int): Request timeout in seconds (default: 10)
            max_retries (int): Maximum retry attempts (default: 3)
        """
        self.api_key = api_key
        self.timeout = timeout
        self.max_retries = max_retries
        
        # Use requests.Session for connection pooling
        self.session = requests.Session()
        
        # Configure session for better performance
        adapter = requests.adapters.HTTPAdapter(
            pool_connections=10,
            pool_maxsize=20,
            max_retries=0  # We handle retries with tenacity
        )
        self.session.mount('http://', adapter)
        self.session.mount('https://', adapter)
        
        self.logger = logging.getLogger('weather_tools.api_client')
        self.logger.info("Weather API Client initialized")
    
    def _log_request(self, endpoint: str, params: Dict[str, Any], duration: float, success: bool):
        """
        Log API request details for observability.
        
        Args:
            endpoint (str): The API endpoint called
            params (Dict[str, Any]): Request parameters (API key will be sanitized)
            duration (float): Request duration in seconds
            success (bool): Whether the request succeeded
        """
        # Sanitize parameters to remove API key
        safe_params = {k: v for k, v in params.items() if k != 'appid'}
        safe_params['appid'] = '***REDACTED***'
        
        status = "SUCCESS" if success else "FAILURE"
        self.logger.info(
            f"API Request [{status}] - Endpoint: {endpoint} - "
            f"Duration: {duration:.2f}s - Params: {safe_params}"
        )
    
    @retry(
        # Retry only on network-related exceptions
        retry=retry_if_exception_type((requests.exceptions.Timeout, 
                                      requests.exceptions.ConnectionError)),
        # Stop after 3 attempts
        stop=stop_after_attempt(3),
        # Exponential backoff: 1s, 2s, 4s with random jitter
        wait=wait_exponential(multiplier=1, min=1, max=10),
        # Log before each retry
        before_sleep=before_sleep_log(logger, logging.WARNING)
    )
    def _make_request(self, endpoint: str, params: Dict[str, Any]) -> Dict[str, Any]:
        """
        Make an API request with automatic retry logic and comprehensive error handling.
        
        Args:
            endpoint (str): The API endpoint URL
            params (Dict[str, Any]): Query parameters for the request
            
        Returns:
            Dict[str, Any]: The JSON response from the API
            
        Raises:
            ValueError: For authentication errors or invalid requests
            ConnectionError: For network-related failures
            TimeoutError: When request exceeds timeout
        """
        start_time = time.time()
        
        try:
            # Add API key to parameters
            params['appid'] = self.api_key
            
            # Make the request with timeout
            response = self.session.get(endpoint, params=params, timeout=self.timeout)
            
            # Calculate duration
            duration = time.time() - start_time
            
            # Handle different HTTP status codes
            if response.status_code == 401:
                self._log_request(endpoint, params, duration, False)
                raise ValueError(
                    "Authentication failed. Please check your OpenWeatherMap API key."
                )
            
            if response.status_code == 404:
                self._log_request(endpoint, params, duration, False)
                raise ValueError(
                    "Location not found. Please check the coordinates or city name."
                )
            
            if response.status_code == 429:
                self._log_request(endpoint, params, duration, False)
                raise ValueError(
                    "API rate limit exceeded. Please try again in a few moments."
                )
            
            # Raise an exception for other error status codes
            response.raise_for_status()
            
            # Parse JSON response
            data = response.json()
            
            # Log successful request
            self._log_request(endpoint, params, duration, True)
            
            return data
            
        except requests.exceptions.Timeout:
            duration = time.time() - start_time
            self._log_request(endpoint, params, duration, False)
            self.logger.error(f"Request timeout after {self.timeout}s")
            raise TimeoutError(
                f"The weather service is taking too long to respond. "
                f"Request timed out after {self.timeout} seconds."
            )
        
        except requests.exceptions.ConnectionError as e:
            duration = time.time() - start_time
            self._log_request(endpoint, params, duration, False)
            self.logger.error(f"Connection error: {str(e)}")
            raise ConnectionError(
                "Unable to connect to the weather service. "
                "Please check your internet connection."
            )
        
        except requests.exceptions.RequestException as e:
            duration = time.time() - start_time
            self._log_request(endpoint, params, duration, False)
            self.logger.error(f"Request failed: {str(e)}")
            raise ValueError(f"Weather API request failed: {str(e)}")
    
    def get_current_weather(self, lat: float, lon: float) -> Dict[str, Any]:
        """
        Get current weather for a location.
        
        Args:
            lat (float): Latitude
            lon (float): Longitude
            
        Returns:
            Dict[str, Any]: Current weather data
        """
        endpoint = "https://api.openweathermap.org/data/2.5/weather"
        params = {
            "lat": lat,
            "lon": lon,
            "units": "metric"  # Use metric units (Celsius, meters/sec)
        }
        return self._make_request(endpoint, params)
    
    def get_forecast(self, lat: float, lon: float) -> Dict[str, Any]:
        """
        Get weather forecast for a location using the FREE tier 5 Day / 3 Hour Forecast API.
        
        This method uses the free-tier forecast endpoint instead of One Call API.
        Returns 3-hour interval forecasts for up to 5 days (40 data points).
        
        Args:
            lat (float): Latitude
            lon (float): Longitude
            
        Returns:
            Dict[str, Any]: Forecast data with 'list' containing forecast points
        """
        endpoint = "https://api.openweathermap.org/data/2.5/forecast"
        params = {
            "lat": lat,
            "lon": lon,
            "units": "metric",  # Use metric units (Celsius, meters/sec)
            "cnt": 16  # Get 16 data points (2 days * 8 forecasts per day)
        }
        return self._make_request(endpoint, params)

# Initialize the API client
api_client = WeatherAPIClient(
    api_key=os.getenv("OPENWEATHER_API_KEY"),
    timeout=10,
    max_retries=3
)

print("\nWeather API Client initialized successfully!")
print("Features: Connection pooling, automatic retries, structured logging")
print("Using FREE tier endpoints: Current Weather API and 5 Day Forecast API")

2025-11-12 16:41:48 - weather_tools.api_client - INFO - Weather API Client initialized



Weather API Client initialized successfully!
Features: Connection pooling, automatic retries, structured logging
Using FREE tier endpoints: Current Weather API and 5 Day Forecast API


## Step 6: Build the Current Weather Tool

**Now we bring it all together!**

This tool combines:
- Pydantic validation (type safety)
- Geocoding (user-friendly city names)
- The API client (resilience and retries)
- Structured logging (observability)
- Clear error messages (user experience)

**Key Details:**

1. The `@tool` decorator makes this function available to LangChain agents
2. The docstring is crucial - the agent reads it to understand when to use this tool
3. We return a human-readable string, not raw JSON
4. All errors are caught and converted to helpful messages

In [21]:
@tool
def get_current_weather(city: str) -> str:
    """
    Get the current weather conditions for a specified city.
    
    Use this tool when the user asks about current weather, present conditions,
    or "right now" weather in any location.
    
    Args:
        city (str): The name of the city (e.g., "London", "New York", "Tokyo")
        
    Returns:
        str: A human-readable description of the current weather including:
             - Temperature (in Celsius)
             - Weather description (e.g., "clear sky", "light rain")
             - Humidity percentage
             - Wind speed (in meters/second)
             
    Example:
        get_current_weather("London") -> "Current weather in London: 15°C, partly cloudy..."
    """
    logger.info(f"Tool invoked: get_current_weather(city={city})")
    start_time = time.time()
    
    try:
        # Step 1: Validate input using Pydantic
        location = WeatherLocation(city=city)
        logger.info(f"Input validation passed for city: {location.city}")
        
        # Step 2: Geocode the city to get coordinates
        lat, lon = geocode_city(location.city)
        
        # Step 3: Fetch current weather data
        weather_data = api_client.get_current_weather(lat, lon)
        
        # Step 4: Format the response for human readability
        temp = weather_data['main']['temp']
        feels_like = weather_data['main']['feels_like']
        description = weather_data['weather'][0]['description']
        humidity = weather_data['main']['humidity']
        wind_speed = weather_data['wind']['speed']
        
        result = (
            f"Current weather in {location.city}:\n"
            f"  Temperature: {temp}°C (feels like {feels_like}°C)\n"
            f"  Conditions: {description.capitalize()}\n"
            f"  Humidity: {humidity}%\n"
            f"  Wind Speed: {wind_speed} m/s"
        )
        
        duration = time.time() - start_time
        logger.info(f"Tool completed successfully in {duration:.2f}s")
        
        return result
        
    except ValueError as e:
        # Input validation or API errors (user-friendly messages)
        duration = time.time() - start_time
        logger.error(f"Validation error after {duration:.2f}s: {str(e)}")
        return f"Error: {str(e)}"
    
    except TimeoutError as e:
        # Timeout errors
        duration = time.time() - start_time
        logger.error(f"Timeout error after {duration:.2f}s: {str(e)}")
        return f"Error: {str(e)}"
    
    except ConnectionError as e:
        # Network errors
        duration = time.time() - start_time
        logger.error(f"Connection error after {duration:.2f}s: {str(e)}")
        return f"Error: {str(e)}"
    
    except Exception as e:
        # Catch-all for unexpected errors
        duration = time.time() - start_time
        logger.error(f"Unexpected error after {duration:.2f}s: {str(e)}", exc_info=True)
        return (
            f"An unexpected error occurred while fetching weather data: {str(e)}. "
            "Please try again or contact support if the problem persists."
        )

# Test the current weather tool
print("\nTesting current weather tool...\n")
result = get_current_weather.invoke({"city": "Singapore"})
print(result)

2025-11-12 16:41:49 - weather_tools - INFO - Tool invoked: get_current_weather(city=Singapore)
2025-11-12 16:41:49 - weather_tools - INFO - Input validation passed for city: Singapore
2025-11-12 16:41:49 - weather_tools - INFO - Geocoding city: Singapore
2025-11-12 16:41:49 - weather_tools - INFO - Successfully geocoded Singapore to (1.2899175, 103.8519072)
2025-11-12 16:41:49 - weather_tools.api_client - INFO - API Request [SUCCESS] - Endpoint: https://api.openweathermap.org/data/2.5/weather - Duration: 0.03s - Params: {'lat': 1.2899175, 'lon': 103.8519072, 'units': 'metric', 'appid': '***REDACTED***'}
2025-11-12 16:41:49 - weather_tools - INFO - Tool completed successfully in 0.05s



Testing current weather tool...

Current weather in Singapore:
  Temperature: 29.33°C (feels like 33.88°C)
  Conditions: Thunderstorm with light rain
  Humidity: 72%
  Wind Speed: 4.12 m/s


## Step 7: Build the Weather Forecast Tool

**The Challenge:**

Weather forecasts return data for multiple days. We need to:
1. Extract only the next 2 days from the forecast
2. Format the data clearly for each day
3. Include relevant forecast information (high/low temps, conditions)

**Production Considerations:**

- The API returns a `daily` array with forecasts for up to 8 days
- We only take the first 2 elements (index 0 and 1 are tomorrow and day after)
- We format both the date and the forecast data clearly
- Forecasts include min/max temperatures, which we display

In [26]:
@tool
def get_weather_forecast(city: str) -> str:
    """
    Get the weather forecast for the next 2 days for a specified city.
    
    Use this tool when the user asks about future weather, weather predictions,
    "tomorrow's weather", or "weather forecast for the next few days".
    
    Args:
        city (str): The name of the city (e.g., "London", "New York", "Tokyo")
        
    Returns:
        str: A forecast summary for the next 2 days including:
             - Date for each day
             - Temperature predictions (in Celsius)
             - Expected weather conditions
             - Humidity and wind predictions
             
    Example:
        get_weather_forecast("Berlin") -> "Weather forecast for Berlin (next 2 days)..."
    """
    logger.info(f"Tool invoked: get_weather_forecast(city={city})")
    start_time = time.time()
    
    try:
        # Step 1: Validate input
        input_data = ForecastWeatherInput(city=city, days=2)
        logger.info(f"Input validation passed for city: {input_data.city}")
        
        # Step 2: Geocode the city
        lat, lon = geocode_city(input_data.city)
        
        # Step 3: Fetch forecast data (free tier 5 Day / 3 Hour Forecast API)
        forecast_data = api_client.get_forecast(lat, lon)
        
        # Step 4: Process the forecast data
        # The API returns forecasts in 3-hour intervals
        # We need to group by day and extract the next 2 days
        
        forecast_list = forecast_data['list']
        
        # Group forecasts by date
        daily_forecasts = defaultdict(list)
        
        for forecast in forecast_list:
            # Convert timestamp to date
            forecast_date = datetime.fromtimestamp(forecast['dt']).date()
            daily_forecasts[forecast_date].append(forecast)
        
        # Get the next 2 days (sorted by date)
        sorted_dates = sorted(daily_forecasts.keys())[:2]
        
        # Step 5: Format the response
        results = []
        
        for i, date in enumerate(sorted_dates):
            day_forecasts = daily_forecasts[date]
            day_label = "tomorrow" if i == 0 else "day after tomorrow"
            
            # Calculate aggregate statistics for the day
            temps = [f['main']['temp'] for f in day_forecasts]
            temp_min = min(f['main']['temp_min'] for f in day_forecasts)
            temp_max = max(f['main']['temp_max'] for f in day_forecasts)
            avg_temp = sum(temps) / len(temps)
            
            # Get the most common weather condition
            conditions = [f['weather'][0]['description'] for f in day_forecasts]
            most_common_condition = max(set(conditions), key=conditions.count)
            
            # Average humidity and wind speed
            avg_humidity = sum(f['main']['humidity'] for f in day_forecasts) / len(day_forecasts)
            avg_wind = sum(f['wind']['speed'] for f in day_forecasts) / len(day_forecasts)
            
            results.append(
                f"  {date.strftime('%Y-%m-%d')} ({day_label}):\n"
                f"    Temperature: {avg_temp:.1f}°C (High: {temp_max:.1f}°C, Low: {temp_min:.1f}°C)\n"
                f"    Conditions: {most_common_condition.capitalize()}\n"
                f"    Humidity: {avg_humidity:.0f}%\n"
                f"    Wind Speed: {avg_wind:.1f} m/s"
            )
        
        result = f"Weather forecast for {input_data.city} (next 2 days):\n\n" + "\n\n".join(results)
        
        duration = time.time() - start_time
        logger.info(f"Tool completed successfully in {duration:.2f}s")
        
        return result
        
    except ValueError as e:
        duration = time.time() - start_time
        logger.error(f"Validation error after {duration:.2f}s: {str(e)}")
        return f"Error: {str(e)}"
    
    except TimeoutError as e:
        duration = time.time() - start_time
        logger.error(f"Timeout error after {duration:.2f}s: {str(e)}")
        return f"Error: {str(e)}"
    
    except ConnectionError as e:
        duration = time.time() - start_time
        logger.error(f"Connection error after {duration:.2f}s: {str(e)}")
        return f"Error: {str(e)}"
    
    except KeyError as e:
        duration = time.time() - start_time
        logger.error(f"Data parsing error after {duration:.2f}s: {str(e)}")
        return (
            f"Error: The weather API returned unexpected data format. "
            f"Missing field: {str(e)}. Please try again."
        )
    
    except Exception as e:
        duration = time.time() - start_time
        logger.error(f"Unexpected error after {duration:.2f}s: {str(e)}", exc_info=True)
        return (
            f"An unexpected error occurred while fetching forecast data: {str(e)}. "
            "Please try again or contact support if the problem persists."
        )

# Test the forecast weather tool
print("\nTesting weather forecast tool...\n")
result = get_weather_forecast.invoke({"city": "Paris"})
print(result)

2025-11-12 16:42:26 - weather_tools - INFO - Tool invoked: get_weather_forecast(city=Paris)
2025-11-12 16:42:26 - weather_tools - INFO - Input validation passed for city: Paris
2025-11-12 16:42:26 - weather_tools - INFO - Geocoding city: Paris
2025-11-12 16:42:26 - weather_tools - INFO - Successfully geocoded Paris to (48.8588897, 2.3200410217200766)
2025-11-12 16:42:26 - weather_tools.api_client - INFO - API Request [SUCCESS] - Endpoint: https://api.openweathermap.org/data/2.5/forecast - Duration: 0.01s - Params: {'lat': 48.8588897, 'lon': 2.3200410217200766, 'units': 'metric', 'cnt': 16, 'appid': '***REDACTED***'}
2025-11-12 16:42:26 - weather_tools - INFO - Tool completed successfully in 0.03s



Testing weather forecast tool...

Weather forecast for Paris (next 2 days):

  2025-11-12 (tomorrow):
    Temperature: 11.8°C (High: 16.3°C, Low: 9.8°C)
    Conditions: Scattered clouds
    Humidity: 70%
    Wind Speed: 3.9 m/s

  2025-11-13 (day after tomorrow):
    Temperature: 14.9°C (High: 17.7°C, Low: 12.4°C)
    Conditions: Overcast clouds
    Humidity: 63%
    Wind Speed: 3.3 m/s


## Step 8: Configure the Language Model

Now we'll set up the LangChain agent that will use our weather tools.

**Key Configuration:**

- **Model**: GPT-4 or GPT-4-Turbo for reliable reasoning and tool usage
- **Temperature**: 0.1 for more consistent and predictable responses

The low temperature ensures the agent makes consistent decisions about which tools to use.

In [27]:
# Configure the language model
model = ChatOpenAI(
    model="gpt-4o",  # Use GPT-4-Turbo for reliable tool usage
    temperature=0.1       # Low temperature for consistent reasoning
)

print(f"Language model configured: {model.model_name}")
print(f"Temperature: {model.temperature}")
print("\nThis model will power the agent's reasoning about which tools to use.")

Language model configured: gpt-4o
Temperature: 0.1

This model will power the agent's reasoning about which tools to use.


## Step 9: Create the Agent with Weather Tools

**The Magic Happens Here:**

We create a LangChain agent and give it access to our 2 weather tools:

1. `get_current_weather` - for present conditions
2. `get_weather_forecast` - for next 2 days

**How It Works:**

The agent will:
1. Read the user's question
2. Examine the docstrings of all available tools
3. Decide which tool(s) to use based on the query
4. Call the appropriate tool(s) with the right parameters
5. Synthesize the results into a natural language response

This is **autonomous decision-making** - we don't tell the agent which tool to use!

In [28]:
# Create a list of our weather tools (2 tools)
weather_tools = [
    get_current_weather,
    get_weather_forecast
]

# Create the agent with access to all weather tools
agent = create_agent(
    model=model,
    tools=weather_tools
)

print("Agent created successfully!")
print(f"\nAgent has access to {len(weather_tools)} tools:")
for tool in weather_tools:
    print(f"  - {tool.name}: {tool.description[:80]}...")

Agent created successfully!

Agent has access to 2 tools:
  - get_current_weather: Get the current weather conditions for a specified city.

Use this tool when the...
  - get_weather_forecast: Get the weather forecast for the next 2 days for a specified city.

Use this too...


## Step 10: Create a Helper Function for Agent Interaction

To make testing easier, we'll create a helper function that:

1. Takes a natural language question
2. Invokes the agent
3. Extracts and prints the final response

This simplifies our test examples and makes the notebook more readable.

In [29]:
def ask_agent(question: str) -> str:
    """
    Ask the weather agent a question and return the response.
    
    Args:
        question (str): The natural language question to ask
        
    Returns:
        str: The agent's response
    """
    print(f"\n{'='*80}")
    print(f"Question: {question}")
    print(f"{'='*80}\n")
    
    # Invoke the agent with properly formatted messages
    result = agent.invoke({
        "messages": [{"role": "user", "content": question}]
    })
    
    # Extract the final response
    final_message = result["messages"][-1]
    response = final_message.content
    
    print(f"Response:\n{response}\n")
    
    return response

print("Helper function created successfully!")
print("Ready to test the agent with natural language questions.")

Helper function created successfully!
Ready to test the agent with natural language questions.


## Step 11: Test the Agent - Current Weather Query

Let's start with a simple query about current weather.

**What to Watch For:**

- The agent should automatically choose `get_current_weather`
- It should extract "London" from the question
- You'll see logging output showing the tool execution
- The final response should be natural and conversational

In [30]:
# Test Example 1: Current weather query
response1 = ask_agent("What's the weather like in London right now?")


Question: What's the weather like in London right now?



2025-11-12 16:43:10 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-12 16:43:10 - weather_tools - INFO - Tool invoked: get_current_weather(city=London)
2025-11-12 16:43:10 - weather_tools - INFO - Input validation passed for city: London
2025-11-12 16:43:10 - weather_tools - INFO - Geocoding city: London
2025-11-12 16:43:10 - weather_tools - INFO - Successfully geocoded London to (51.5073219, -0.1276474)
2025-11-12 16:43:10 - weather_tools.api_client - INFO - API Request [SUCCESS] - Endpoint: https://api.openweathermap.org/data/2.5/weather - Duration: 0.01s - Params: {'lat': 51.5073219, 'lon': -0.1276474, 'units': 'metric', 'appid': '***REDACTED***'}
2025-11-12 16:43:10 - weather_tools - INFO - Tool completed successfully in 0.36s
2025-11-12 16:43:12 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Response:
The current weather in London is overcast with a temperature of 14.15°C, which feels like 13.79°C. The humidity is at 83%, and the wind speed is 3.13 meters per second.



## Step 12: Test the Agent - Forecast Query

Let's test the forecast tool.

**What to Watch For:**

- The agent should choose `get_weather_forecast` based on "next 2 days"
- The response should include high/low temperatures for each day
- Watch the structured logging show execution time and success status

In [31]:
# Test Example 2: Weather forecast query
response2 = ask_agent("What's the weather forecast for Singapore for the next 2 days?")


Question: What's the weather forecast for Singapore for the next 2 days?



2025-11-12 16:43:30 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-12 16:43:30 - weather_tools - INFO - Tool invoked: get_weather_forecast(city=Singapore)
2025-11-12 16:43:30 - weather_tools - INFO - Input validation passed for city: Singapore
2025-11-12 16:43:30 - weather_tools - INFO - Geocoding city: Singapore
2025-11-12 16:43:30 - weather_tools - INFO - Successfully geocoded Singapore to (1.2899175, 103.8519072)
2025-11-12 16:43:30 - weather_tools.api_client - INFO - API Request [SUCCESS] - Endpoint: https://api.openweathermap.org/data/2.5/forecast - Duration: 0.02s - Params: {'lat': 1.2899175, 'lon': 103.8519072, 'units': 'metric', 'cnt': 16, 'appid': '***REDACTED***'}
2025-11-12 16:43:30 - weather_tools - INFO - Tool completed successfully in 0.04s
2025-11-12 16:43:32 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Response:
The weather forecast for Singapore for the next 2 days is as follows:

- **Tomorrow (2025-11-12):**
  - Temperature: 28.7°C (High: 29.5°C, Low: 26.7°C)
  - Conditions: Broken clouds
  - Humidity: 73%
  - Wind Speed: 5.2 m/s

- **Day after tomorrow (2025-11-13):**
  - Temperature: 27.1°C (High: 28.3°C, Low: 25.9°C)
  - Conditions: Light rain
  - Humidity: 80%
  - Wind Speed: 3.8 m/s



## Step 13: Test the Agent - Multi-City Comparison

Now let's test something more complex: comparing weather across multiple cities.

**What to Watch For:**

- The agent should call `get_current_weather` **twice** (once for Paris, once for Berlin)
- It should synthesize the results into a comparison
- This demonstrates the agent's ability to orchestrate multiple tool calls
- Each city's geocoding and API call will be logged separately

In [32]:
# Test Example 3: Multi-city comparison
response3 = ask_agent("Compare the current weather in Paris and Berlin")


Question: Compare the current weather in Paris and Berlin



2025-11-12 16:43:58 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-12 16:43:58 - weather_tools - INFO - Tool invoked: get_current_weather(city=Paris)
2025-11-12 16:43:58 - weather_tools - INFO - Tool invoked: get_current_weather(city=Berlin)
2025-11-12 16:43:58 - weather_tools - INFO - Input validation passed for city: Berlin
2025-11-12 16:43:58 - weather_tools - INFO - Geocoding city: Berlin
2025-11-12 16:43:58 - weather_tools - INFO - Input validation passed for city: Paris
2025-11-12 16:43:58 - weather_tools - INFO - Geocoding city: Paris
2025-11-12 16:43:58 - weather_tools - INFO - Successfully geocoded Paris to (48.8588897, 2.3200410217200766)
2025-11-12 16:43:58 - weather_tools.api_client - INFO - API Request [SUCCESS] - Endpoint: https://api.openweathermap.org/data/2.5/weather - Duration: 0.01s - Params: {'lat': 48.8588897, 'lon': 2.3200410217200766, 'units': 'metric', 'appid': '***REDACTED***'}
2025-11-12 16:43:58 - weat

Response:
Here's the current weather comparison between Paris and Berlin:

**Paris:**
- Temperature: 9.06°C (feels like 6.79°C)
- Conditions: Clear sky
- Humidity: 85%
- Wind Speed: 4.12 m/s

**Berlin:**
- Temperature: 7.09°C (feels like 4.93°C)
- Conditions: Clear sky
- Humidity: 89%
- Wind Speed: 3.13 m/s

Both cities are experiencing clear skies, but Paris is slightly warmer than Berlin.



In [33]:
response3 = ask_agent("Is it hotter in Singapore or Phuket today?")


Question: Is it hotter in Singapore or Phuket today?



2025-11-12 16:47:19 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-12 16:47:19 - weather_tools - INFO - Tool invoked: get_current_weather(city=Singapore)
2025-11-12 16:47:19 - weather_tools - INFO - Tool invoked: get_current_weather(city=Phuket)
2025-11-12 16:47:19 - weather_tools - INFO - Input validation passed for city: Phuket
2025-11-12 16:47:19 - weather_tools - INFO - Geocoding city: Phuket
2025-11-12 16:47:19 - weather_tools - INFO - Input validation passed for city: Singapore
2025-11-12 16:47:19 - weather_tools - INFO - Geocoding city: Singapore
2025-11-12 16:47:19 - weather_tools - INFO - Successfully geocoded Singapore to (1.2899175, 103.8519072)
2025-11-12 16:47:19 - weather_tools - INFO - Successfully geocoded Phuket to (7.8847901, 98.3891503)
2025-11-12 16:47:19 - weather_tools.api_client - INFO - API Request [SUCCESS] - Endpoint: https://api.openweathermap.org/data/2.5/weather - Duration: 0.05s - Params: {'lat': 7.

Response:
Today, the temperature in Singapore is 29.52°C, while in Phuket, it is slightly warmer at 30.06°C.



In [35]:
# Test Example 4: Invalid city name (error handling)
response4 = ask_agent("What's the weather in xyz?")


Question: What's the weather in xyz?



2025-11-12 16:47:45 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Response:
It seems like "xyz" might be a placeholder or an error in the city name. Could you please provide the correct name of the city you're interested in?



## Step 14: Test Error Handling - Simulating Production Error Conditions

Now let's test our production-grade error handling by simulating various failure scenarios. This demonstrates how the system **gracefully degrades** instead of crashing.

**Error Scenarios We'll Test:**

1. **Invalid City Name** - Non-existent location
2. **Malicious Input** - SQL injection attempt
3. **Empty/Whitespace Input** - Validation failure
4. **Special Characters** - Security validation
5. **Numbers as City Name** - Type confusion

**What to Watch For:**

- ✅ System never crashes
- ✅ Clear, user-friendly error messages
- ✅ Detailed logging for debugging
- ✅ Input sanitization blocks malicious inputs
- ✅ Validation catches invalid data early

This is the difference between a basic implementation and a **production-ready system**!

In [36]:
print("="*80)
print("ERROR CONDITION TESTING - Production-Grade Error Handling")
print("="*80)

# Error Test 1: Non-existent City
print("\n1. Testing Invalid City Name (Non-existent location)")
print("-" * 80)
try:
    result = get_current_weather.invoke({"city": "XYZ123InvalidCity"})
    print(f"Result: {result}")
except Exception as e:
    print(f"Exception caught: {e}")

# Error Test 2: SQL Injection Attempt
print("\n2. Testing Malicious Input (SQL Injection Attempt)")
print("-" * 80)
try:
    result = get_current_weather.invoke({"city": "London'; DROP TABLE weather;--"})
    print(f"Result: {result}")
except Exception as e:
    print(f"Exception caught: {e}")

# Error Test 3: Empty/Whitespace Input
print("\n3. Testing Empty/Whitespace Input (Validation Failure)")
print("-" * 80)
try:
    result = get_current_weather.invoke({"city": "    "})
    print(f"Result: {result}")
except Exception as e:
    print(f"Exception caught: {e}")

# Error Test 4: Special Characters
print("\n4. Testing Special Characters (Security Validation)")
print("-" * 80)
try:
    result = get_current_weather.invoke({"city": "<script>alert('XSS')</script>"})
    print(f"Result: {result}")
except Exception as e:
    print(f"Exception caught: {e}")

# Error Test 5: Numbers Only
print("\n5. Testing Numbers as City Name (Type Confusion)")
print("-" * 80)
try:
    result = get_current_weather.invoke({"city": "12345"})
    print(f"Result: {result}")
except Exception as e:
    print(f"Exception caught: {e}")

# Error Test 6: Extremely Long Input
print("\n6. Testing Extremely Long City Name (Input Length Validation)")
print("-" * 80)
try:
    result = get_current_weather.invoke({"city": "A" * 200})
    print(f"Result: {result}")
except Exception as e:
    print(f"Exception caught: {e}")

print("\n" + "="*80)
print("SUMMARY: All error conditions handled gracefully!")
print("="*80)
print("\nKey Observations:")
print("✅ No system crashes or unhandled exceptions")
print("✅ Clear, actionable error messages for users")
print("✅ Detailed logging for debugging (check logs above)")
print("✅ Input sanitization blocked malicious inputs")
print("✅ Validation caught invalid data before API calls")
print("\nProduction-Grade Error Handling: VERIFIED ✓")

2025-11-12 16:58:34 - weather_tools - INFO - Tool invoked: get_current_weather(city=XYZ123InvalidCity)
2025-11-12 16:58:34 - weather_tools - INFO - Input validation passed for city: XYZ123InvalidCity
2025-11-12 16:58:34 - weather_tools - INFO - Geocoding city: XYZ123InvalidCity


ERROR CONDITION TESTING - Production-Grade Error Handling

1. Testing Invalid City Name (Non-existent location)
--------------------------------------------------------------------------------


2025-11-12 16:58:34 - weather_tools - WARNING - City not found: XYZ123InvalidCity
2025-11-12 16:58:34 - weather_tools - ERROR - Validation error after 0.23s: Could not find coordinates for 'XYZ123InvalidCity'. Please check the spelling or try a different city name.
2025-11-12 16:58:34 - weather_tools - INFO - Tool invoked: get_current_weather(city=London'; DROP TABLE weather;--)
2025-11-12 16:58:34 - weather_tools - ERROR - Validation error after 0.00s: 1 validation error for WeatherLocation
city
  Value error, City name contains invalid characters. Only letters, numbers, spaces, hyphens, and apostrophes are allowed. [type=value_error, input_value="London'; DROP TABLE weather;--", input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error
2025-11-12 16:58:34 - weather_tools - INFO - Tool invoked: get_current_weather(city=    )
2025-11-12 16:58:34 - weather_tools - ERROR - Validation error after 0.00s: 1 validation error for WeatherLocation
city
  V

Result: Error: Could not find coordinates for 'XYZ123InvalidCity'. Please check the spelling or try a different city name.

2. Testing Malicious Input (SQL Injection Attempt)
--------------------------------------------------------------------------------
Result: Error: 1 validation error for WeatherLocation
city
  Value error, City name contains invalid characters. Only letters, numbers, spaces, hyphens, and apostrophes are allowed. [type=value_error, input_value="London'; DROP TABLE weather;--", input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

3. Testing Empty/Whitespace Input (Validation Failure)
--------------------------------------------------------------------------------
Result: Error: 1 validation error for WeatherLocation
city
  Value error, City name cannot be empty or just whitespace [type=value_error, input_value='    ', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

4. Test

2025-11-12 16:58:35 - weather_tools - WARNING - City not found: 12345
2025-11-12 16:58:35 - weather_tools - ERROR - Validation error after 0.21s: Could not find coordinates for '12345'. Please check the spelling or try a different city name.
2025-11-12 16:58:35 - weather_tools - INFO - Tool invoked: get_current_weather(city=AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA)
2025-11-12 16:58:35 - weather_tools - ERROR - Validation error after 0.00s: 1 validation error for WeatherLocation
city
  String should have at most 100 characters [type=string_too_long, input_value='AAAAAAAAAAAAAAAAAAAAAAAA...AAAAAAAAAAAAAAAAAAAAAAA', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/string_too_long


Result: Error: Could not find coordinates for '12345'. Please check the spelling or try a different city name.

6. Testing Extremely Long City Name (Input Length Validation)
--------------------------------------------------------------------------------
Result: Error: 1 validation error for WeatherLocation
city
  String should have at most 100 characters [type=string_too_long, input_value='AAAAAAAAAAAAAAAAAAAAAAAA...AAAAAAAAAAAAAAAAAAAAAAA', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/string_too_long

SUMMARY: All error conditions handled gracefully!

Key Observations:
✅ No system crashes or unhandled exceptions
✅ Clear, actionable error messages for users
✅ Detailed logging for debugging (check logs above)
✅ Input sanitization blocked malicious inputs
✅ Validation caught invalid data before API calls

Production-Grade Error Handling: VERIFIED ✓
